# Тренировка моделей (Colab / Kaggle)

Воспроизведение экспериментов из статьи "Why Can't Transformers Learn Multiplication?"

## 0. Setup

In [ ]:
# Клонируем репозиторий (замени URL на свой)
!git clone https://github.com/YOUR_USERNAME/diplom.git /content/diplom
%cd /content/diplom

# Установка зависимостей
!pip install -q -r requirements.txt

In [ ]:
# Проверка GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Генерация данных

In [ ]:
!python generate_data.py

## 2. Тренировка ICoT

2 слоя, 4 головы, n_embd=768 (~54M параметров). Ожидаемо 100% accuracy после ~13 эпох.

In [ ]:
!TOKENIZERS_PARALLELISM=false python Internalize_CoT_Step_by_Step/src/train.py \
    --model gpt2 --n_layer 2 --n_head 4 --n_embd 768 \
    --train_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/train.txt \
    --val_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/valid.txt \
    --lr 5e-5 --batch_size 32 --seed 3456 --reset_optimizer \
    --epochs 200 \
    --remove_per_epoch 8 \
    --remove_all_when_remove_beyond inf \
    --removal_smoothing_lambda 4 \
    --removal_side left \
    --pretrain_epochs 0 \
    --save_model train_models/4_by_4_mult/icot

## 3. Тренировка SFT

Без CoT (все токены удалены сразу). Ожидаемо <1% accuracy.

In [ ]:
!TOKENIZERS_PARALLELISM=false python Internalize_CoT_Step_by_Step/src/train.py \
    --model gpt2 --n_layer 2 --n_head 4 --n_embd 768 \
    --train_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/train.txt \
    --val_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/valid.txt \
    --lr 5e-5 --batch_size 32 --seed 3456 --reset_optimizer \
    --epochs 60 \
    --remove_per_epoch 99999 \
    --removal_side left \
    --save_model train_models/4_by_4_mult/sft

## 4. Инференс

In [ ]:
# ICoT inference
!TOKENIZERS_PARALLELISM=false python Internalize_CoT_Step_by_Step/src/generate.py \
    --from_pretrained train_models/4_by_4_mult/icot/checkpoint_12 \
    --test_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/test_bigbench.txt \
    --max_new_tokens 800

## 5. Скачать чекпоинты (опционально)

Сохранить результаты на Google Drive, чтобы не потерять при перезапуске Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r train_models /content/drive/MyDrive/diplom_checkpoints/